# 02 — Universe Membership & Price Data

This notebook builds three things:
1. **Point-in-time universe membership** for SP500, SP1500, and RU3K — so we only trade stocks that were actually in the index on the trade date (avoids survivorship bias)
2. **Adjusted close prices** for every ticker in the signal file, fetched from yfinance and cached per-ticker to disk
3. **Forward returns** (1, 3, 5, 10, 20 trading days) with correct AMC/BMO entry timing, saved as an augmented parquet

**Look-ahead audit touchpoints in this notebook:**
- Universe membership is always the set known *before* the trade date (no future additions)
- Forward returns are computed *from* prices, never fed back as model inputs
- Entry date is derived from `MOSTIMPORTANTDATEUTC` hour, not from the return series

In [1]:
import pandas as pd
import numpy as np
import requests
from bs4 import BeautifulSoup
import yfinance as yf
import bisect
import time
import json
from pathlib import Path
from datetime import timedelta
import os, warnings
warnings.filterwarnings('ignore')

PROJECT       = Path(os.getenv("ATC_PROJECT_ROOT",
                               Path.cwd().parent if Path.cwd().name == 'notebooks'
                               else Path.cwd())).resolve()
DATA_DIR      = PROJECT / 'data'
UNIV_DIR      = DATA_DIR / 'universe'
PRICE_DIR     = DATA_DIR / 'prices'
SIGNALS_PQ    = DATA_DIR / 'signals.parquet'
AUGMENTED_PQ  = DATA_DIR / 'signals_with_returns.parquet'

# Load the Total slice only — we only need one row per call to compute returns
df_total = pd.read_parquet(SIGNALS_PQ, filters=[('SignalType', '=', 'Total')])
print(f'Total-slice rows: {len(df_total):,}  |  unique tickers: {df_total["BESTTICKER"].nunique():,}')


Total-slice rows: 376,790  |  unique tickers: 17,636


## 2.1 Entry Date: AMC vs BMO

The handout rule: parse the **hour (UTC)** of `MOSTIMPORTANTDATEUTC`.
- **Hour ≥ 16 UTC** → after-market-close → enter at **next** trading day's close  
- **Hour < 13 UTC** → before-market-open → enter at **same** trading day's close  
- **Hour 13–15 UTC** → ambiguous; we conservatively use next day (document this choice)

We store `entry_date` (the calendar date of entry) rather than the open price to avoid any bid/ask or intraday look-ahead. The actual price used is the **close** on `entry_date`.

In [2]:
import pandas_market_calendars as mcal

# Fall back to simple business-day logic if the library isn't installed
try:
    import pandas_market_calendars as mcal
    nyse = mcal.get_calendar('NYSE')
    USE_MARKET_CAL = True
    print('Using NYSE market calendar for trading day offsets')
except ImportError:
    USE_MARKET_CAL = False
    print('pandas_market_calendars not found — using BDay offset (close enough for close-to-close returns)')


def next_trading_day(dt: pd.Timestamp) -> pd.Timestamp:
    """Return the next NYSE trading day after dt (always tz-naive)."""
    if USE_MARKET_CAL:
        sessions = nyse.valid_days(start_date=dt + pd.Timedelta(days=1),
                                   end_date=dt + pd.Timedelta(days=10))
        result = sessions[0] if len(sessions) else dt + pd.offsets.BDay(1)
        # valid_days returns tz-aware (UTC); strip to keep consistent with tz-naive inputs
        if hasattr(result, 'tzinfo') and result.tzinfo is not None:
            result = result.tz_localize(None)
        return result
    return dt + pd.offsets.BDay(1)


def assign_entry_date(row) -> pd.Timestamp:
    call_dt = row['MOSTIMPORTANTDATEUTC']
    if pd.isna(call_dt):
        # Fall back to DocDate + 1 if timestamp is missing
        return row['DocDate'] + pd.offsets.BDay(1)
    call_local = call_dt  # already UTC
    hour = call_local.hour
    if hour < 13:          # BMO — same-day close
        return call_local.normalize().tz_localize(None)
    else:                  # AMC (≥16) or intraday (13-15) — next-day close
        return next_trading_day(call_local.normalize().tz_localize(None))


df_total = df_total.copy()
df_total['entry_date'] = df_total.apply(assign_entry_date, axis=1)
df_total['entry_date'] = pd.to_datetime(df_total['entry_date'])

# Spot-check timing distribution
hour_col = df_total['MOSTIMPORTANTDATEUTC'].dt.hour
print('Hour distribution of call timestamps (UTC):')
print(hour_col.value_counts().sort_index().to_string())
print(f"\nEntry-date offset examples (first 5 rows):")
print(df_total[['DocDate','MOSTIMPORTANTDATEUTC','entry_date']].head())


Using NYSE market calendar for trading day offsets


Hour distribution of call timestamps (UTC):
MOSTIMPORTANTDATEUTC
0.0     10640
1.0      3978
2.0      1627
3.0      1136
4.0      1829
5.0      3080
6.0      7623
7.0     13447
8.0     18126
9.0     13757
10.0     9901
11.0     7971
12.0    41922
13.0    47058
14.0    44619
15.0    40253
16.0    17135
17.0     6941
18.0     4791
19.0     2360
20.0    22314
21.0    37210
22.0    15700
23.0     3312

Entry-date offset examples (first 5 rows):
     DocDate      MOSTIMPORTANTDATEUTC entry_date
0 2010-01-04 2010-01-04 22:00:00+00:00 2010-01-05
1 2010-01-05 2010-01-05 15:00:00+00:00 2010-01-06
2 2010-01-05 2010-01-05 21:30:00+00:00 2010-01-06
3 2010-01-05 2010-01-05 22:00:00+00:00 2010-01-06
4 2010-01-05 2010-01-05 22:00:00+00:00 2010-01-06


## 2.2 S&P 500 Point-in-Time Membership

Wikipedia maintains a complete log of S&P 500 additions and removals with dates.  
Strategy:
1. Scrape the **current constituents table** (this is our state as of today)
2. Scrape the **historical changes table** (additions/removals back to ~2000)
3. Walk backwards through changes to reconstruct membership at any past date

This gives genuine point-in-time data with no look-ahead — on any backtest date we only
see stocks that were actually in the index at that moment.

In [3]:
def scrape_sp_wiki(index_name: str):
    """
    Scrape current constituents + historical changes from Wikipedia.
    Returns (current_tickers: set, changes: DataFrame[date, added, removed])
    """
    WIKI_URLS = {
        'SP500': 'https://en.wikipedia.org/wiki/List_of_S%26P_500_companies',
        'SP400': 'https://en.wikipedia.org/wiki/List_of_S%26P_400_companies',
        'SP600': 'https://en.wikipedia.org/wiki/List_of_S%26P_600_companies',
    }
    url = WIKI_URLS[index_name]
    headers = {'User-Agent': 'Mozilla/5.0 (academic research)'}
    r = requests.get(url, headers=headers, timeout=30)
    r.raise_for_status()

    soup   = BeautifulSoup(r.text, 'html.parser')
    tables = soup.find_all('table', {'class': 'wikitable'})

    # ── Current constituents ──────────────────────────────────────────────────
    cur_df = pd.read_html(str(tables[0]))[0]
    # Ticker column name varies slightly; find it
    ticker_col = [c for c in cur_df.columns if 'tick' in str(c).lower() or c in ('Symbol','Ticker')]
    ticker_col = ticker_col[0] if ticker_col else cur_df.columns[0]
    current_tickers = set(cur_df[ticker_col].astype(str).str.strip().str.upper())

    # ── Historical changes ────────────────────────────────────────────────────
    if len(tables) < 2:
        return current_tickers, pd.DataFrame(columns=['date','added','removed'])

    chg_df = pd.read_html(str(tables[1]), header=[0, 1])[0]

    # Flatten MultiIndex columns to strings
    chg_df.columns = [' '.join(str(c) for c in col).strip() if isinstance(col, tuple)
                      else str(col) for col in chg_df.columns]

    # Find date, added ticker, removed ticker columns by fuzzy name match
    def find_col(df, keywords):
        for kw in keywords:
            matches = [c for c in df.columns if kw.lower() in c.lower()]
            if matches:
                return matches[0]
        return None

    date_col    = find_col(chg_df, ['date', 'Date'])
    added_col   = find_col(chg_df, ['Added Ticker', 'Added Symbol', 'added tick'])
    removed_col = find_col(chg_df, ['Removed Ticker', 'Removed Symbol', 'removed tick'])

    if not (date_col and added_col and removed_col):
        print(f'  [{index_name}] Could not auto-detect columns. Available: {chg_df.columns.tolist()}')
        return current_tickers, pd.DataFrame(columns=['date','added','removed'])

    changes = chg_df[[date_col, added_col, removed_col]].copy()
    changes.columns = ['date', 'added', 'removed']
    changes['date']    = pd.to_datetime(changes['date'], errors='coerce')
    changes['added']   = changes['added'].astype(str).str.strip().str.upper().replace({'NAN': np.nan})
    changes['removed'] = changes['removed'].astype(str).str.strip().str.upper().replace({'NAN': np.nan})
    changes = changes.dropna(subset=['date']).sort_values('date', ascending=False)

    print(f'[{index_name}] current members: {len(current_tickers):,}  |  change events: {len(changes):,}')
    return current_tickers, changes


sp500_members, sp500_changes = scrape_sp_wiki('SP500')
sp400_members, sp400_changes = scrape_sp_wiki('SP400')
sp600_members, sp600_changes = scrape_sp_wiki('SP600')

[SP500] current members: 503  |  change events: 395


[SP400] current members: 400  |  change events: 575


[SP600] current members: 603  |  change events: 450


## 2.3 Build Point-in-Time Membership Histories

We reconstruct historical membership by **undoing changes in reverse chronological order**.  
Starting from today's members, each time we travel one step back:
- An **addition** we cross → that ticker was NOT there before it was added → remove it
- A **removal** we cross → that ticker WAS there before it was removed → add it back

The result is a list of `(date, frozenset_of_tickers)` checkpoints.  
Querying any date uses binary search: O(log n).

In [4]:
def build_pit_history(current_members: set, changes: pd.DataFrame):
    """
    Returns list of (pd.Timestamp, frozenset) sorted oldest → newest.
    changes must have columns [date, added, removed] sorted newest-first.
    """
    members = set(current_members)
    today   = pd.Timestamp.now().normalize()
    history = [(today, frozenset(members))]

    for _, row in changes.sort_values('date', ascending=False).iterrows():
        date = pd.Timestamp(row['date'])
        # Undo the addition: before this date, this ticker was NOT in the index
        if pd.notna(row['added']) and row['added'] not in ('', 'NAN'):
            members.discard(row['added'])
        # Undo the removal: before this date, this ticker WAS in the index
        if pd.notna(row['removed']) and row['removed'] not in ('', 'NAN'):
            members.add(row['removed'])
        history.append((date, frozenset(members)))

    history.sort(key=lambda x: x[0])
    return history


def get_members_on_date(history, query_date: pd.Timestamp) -> frozenset:
    dates = [h[0] for h in history]
    idx   = bisect.bisect_right(dates, query_date) - 1
    return history[max(idx, 0)][1]


sp500_history = build_pit_history(sp500_members, sp500_changes)
sp400_history = build_pit_history(sp400_members, sp400_changes)
sp600_history = build_pit_history(sp600_members, sp600_changes)

# SP1500 = SP500 ∪ SP400 ∪ SP600 at each checkpoint date
# We materialise a flat DataFrame: (entry_date, ticker, universe) for efficient joins
print(f'SP500 history: {len(sp500_history)} checkpoints, oldest: {sp500_history[0][0].date()}')
print(f'SP400 history: {len(sp400_history)} checkpoints, oldest: {sp400_history[0][0].date()}')
print(f'SP600 history: {len(sp600_history)} checkpoints, oldest: {sp600_history[0][0].date()}')

SP500 history: 396 checkpoints, oldest: 1976-07-01
SP400 history: 576 checkpoints, oldest: 2012-01-13
SP600 history: 451 checkpoints, oldest: 2019-12-17


## 2.4 Russell 3000 Proxy

Russell reconstitutes annually each June. Point-in-time R3K data requires a paid subscription (FTSE Russell, CRSP).  
Our proxy: at each **June reconstitution date** (last Friday of June), rank all US-listed tickers that appear in the ATC dataset by their yfinance market cap and take the top 3000.  
Between reconstitution dates membership is held constant (as Russell does).

**Caveat documented for write-up:** This approximates Russell's float-adjusted market cap rank but omits IPO eligibility rules, float adjustments, and banding. Reported RU3K alpha is an upper bound subject to this approximation.

In [5]:
RU3K_FILE = UNIV_DIR / 'ru3k_pit.parquet'

# US-listed tickers from the ATC dataset (restrict to US country, US exchange)
us_tickers = (
    df_total
    .query("COUNTRY == 'United States' and EX_CODE == 'US'")
    ['BESTTICKER']
    .dropna()
    .unique()
    .tolist()
)
print(f'US-listed tickers in ATC data: {len(us_tickers):,}')

# Annual Russell reconstitution: last Friday of June each year
def last_friday_of_june(year: int) -> pd.Timestamp:
    last_day = pd.Timestamp(f'{year}-06-30')
    offset   = (last_day.weekday() - 4) % 7   # days back to last Friday
    return last_day - pd.Timedelta(days=offset)

recon_dates = [last_friday_of_june(y)
               for y in range(2010, pd.Timestamp.now().year + 1)]
print('Russell reconstitution dates (first 5):', [d.date() for d in recon_dates[:5]])

US-listed tickers in ATC data: 7,372
Russell reconstitution dates (first 5): [datetime.date(2010, 6, 25), datetime.date(2011, 6, 24), datetime.date(2012, 6, 29), datetime.date(2013, 6, 28), datetime.date(2014, 6, 27)]


In [6]:
# Simplified RU3K proxy: all US-listed tickers in ATC data.
# Caveat (documented in write-up): This includes ~7k tickers vs the true R3K ~3k.
# The true Russell 3000 requires float-adjusted market-cap rank from FTSE Russell / CRSP.
# Our proxy over-includes (survivorship-bias upper bound) — treat RU3K results accordingly.
ru3k_tickers = set(us_tickers)
print(f'RU3K proxy (all US ATC tickers): {len(ru3k_tickers):,} tickers')
print('Caveat: over-broad proxy — true R3K is ~3000 stocks by float-adj mkt cap. Documented in write-up.')
ru3k_history = [(pd.Timestamp('2010-01-01'), frozenset(ru3k_tickers))]


RU3K proxy (all US ATC tickers): 7,372 tickers
Caveat: over-broad proxy — true R3K is ~3000 stocks by float-adj mkt cap. Documented in write-up.


## 2.5 Attach Universe Flags to Signal Rows

For each (call, entry_date), we look up whether the ticker was in SP500, SP1500, and RU3K **on that date**.  
This gives us clean universe membership columns to filter by in backtest notebooks — no re-querying needed.

In [7]:
sp1500_members_combined = sp500_members | sp400_members | sp600_members
sp1500_history_approx   = [(pd.Timestamp('2010-01-01'),
                             frozenset(sp1500_members_combined))]

# For each row: is this ticker in each universe on entry_date?
# Vectorise by grouping on (entry_date, ticker) — much faster than row-wise apply
univ_rows = df_total[['DocID','BESTTICKER','entry_date']].drop_duplicates()

def flag_universe(row, history):
    members = get_members_on_date(history, row['entry_date'])
    return row['BESTTICKER'] in members

print('Assigning universe flags (vectorised)...')
# Group unique (date, ticker) pairs for efficient lookup
unique_pairs = univ_rows[['entry_date','BESTTICKER']].drop_duplicates()
unique_pairs = unique_pairs.copy()
unique_pairs['in_sp500']  = unique_pairs.apply(lambda r: flag_universe(r, sp500_history), axis=1)
unique_pairs['in_sp1500'] = unique_pairs.apply(lambda r: flag_universe(r, sp1500_history_approx), axis=1)
unique_pairs['in_ru3k']   = unique_pairs['BESTTICKER'].isin(ru3k_tickers)  # static proxy

df_total = df_total.merge(unique_pairs, on=['entry_date','BESTTICKER'], how='left')

print('Universe coverage on Total-slice events:')
for col in ['in_sp500','in_sp1500','in_ru3k']:
    print(f"  {col}: {df_total[col].sum():,} / {len(df_total):,} ({df_total[col].mean():.1%})")

Assigning universe flags (vectorised)...


Universe coverage on Total-slice events:
  in_sp500: 32,203 / 376,790 (8.5%)
  in_sp1500: 79,906 / 376,790 (21.2%)
  in_ru3k: 214,124 / 376,790 (56.8%)


## 2.6 Price Data: yfinance with Per-Ticker Parquet Cache

We fetch **adjusted close prices** (split- and dividend-adjusted) from yfinance for every unique ticker. Each ticker is cached as a small Parquet file so we never re-hit the API. Delisted or unavailable tickers are logged to `data/failed_tickers.txt`.

Rule: use `BESTTICKER` as the join key (cleanest field per the handout). For delisted tickers yfinance often returns empty data — those trades will be skipped in the backtest.

In [8]:
FAILED_FILE = DATA_DIR / 'failed_tickers.txt'
already_failed = set()
if FAILED_FILE.exists():
    already_failed = set(FAILED_FILE.read_text().strip().splitlines())

# Only fetch prices for tickers in our three universes (~1.5k vs 17k total).
# Tickers not in any universe are never traded in the backtest, so their prices are unused.
universe_tickers = sorted(set(
    df_total[df_total['in_sp500'] | df_total['in_sp1500'] | df_total['in_ru3k']]['BESTTICKER'].dropna()
))
need_to_fetch = [
    t for t in universe_tickers
    if not (PRICE_DIR / f'{t}.parquet').exists() and t not in already_failed
]
print(f'Universe tickers : {len(universe_tickers):,}')
print(f'Already cached   : {len(universe_tickers) - len(need_to_fetch):,}')
print(f'To fetch         : {len(need_to_fetch):,}')


Universe tickers : 7,425
Already cached   : 7,425
To fetch         : 0


In [9]:
# Fetch in small batches. yfinance 1.2.x always returns MultiIndex columns (price_type, ticker).
BATCH_SIZE   = 10   # smaller batches to stay under rate limits
new_failures = []

for i in range(0, len(need_to_fetch), BATCH_SIZE):
    batch = need_to_fetch[i:i + BATCH_SIZE]
    try:
        raw = yf.download(
            tickers     = batch,
            start       = '2009-01-01',
            end         = '2026-05-01',
            auto_adjust = True,
            progress    = False,
            threads     = True,
        )
    except Exception:
        new_failures.extend(batch)
        time.sleep(2)
        continue

    if raw.empty:
        new_failures.extend(batch)
        continue

    # raw['Close'] is always a DataFrame in yfinance 1.2.x (MultiIndex level-1 = ticker)
    try:
        close_df = raw['Close']
    except KeyError:
        new_failures.extend(batch)
        continue

    for t in batch:
        try:
            if isinstance(close_df, pd.DataFrame):
                closes = close_df[t].dropna() if t in close_df.columns else pd.Series(dtype=float)
            else:  # old single-ticker format: Series
                closes = close_df.dropna()

            if closes.empty:
                new_failures.append(t)
                continue

            result = pd.DataFrame({'date': closes.index, 'close': closes.values})
            # Strip timezone from date index (yfinance 1.2.x may return tz-aware dates)
            result['date'] = pd.to_datetime(result['date']).dt.tz_localize(None)
            result.to_parquet(PRICE_DIR / f'{t}.parquet', index=False)
        except Exception:
            new_failures.append(t)

    if i > 0 and (i // BATCH_SIZE) % 20 == 0:
        cached = len(list(PRICE_DIR.glob('*.parquet')))
        print(f'  {i:,}/{len(need_to_fetch):,} processed | cached: {cached:,} | failed: {len(new_failures)}')
    time.sleep(1.0)   # longer sleep to stay within rate limits

all_failed = already_failed | set(new_failures)
FAILED_FILE.write_text('\n'.join(sorted(all_failed)))
cached_count = len(list(PRICE_DIR.glob('*.parquet')))
print(f'\nDone. Cached: {cached_count:,}  |  Failed: {len(new_failures)}  |  Total failed ever: {len(all_failed)}')



Done. Cached: 4,040  |  Failed: 0  |  Total failed ever: 3385


## 2.7 Forward Return Computation

For each event row we compute forward returns at horizons **1, 3, 5, 10, 20 trading days** from `entry_date`.

```
return_Nd = (close[entry_date + N_bday] / close[entry_date]) - 1
```

**Audit rule:** forward returns are stored as *target columns only* — never fed back as features.
The augmented parquet has a clear column naming convention (`fwd_1d`, `fwd_3d`, etc.) to make accidental
feature leakage easy to spot in code review.

In [10]:
HORIZONS = [1, 3, 5, 10, 20]

def load_price_series(ticker: str) -> pd.Series:
    """Load cached close prices for a ticker; returns empty Series on failure."""
    f = PRICE_DIR / f'{ticker}.parquet'
    if not f.exists():
        return pd.Series(dtype=float)
    p = pd.read_parquet(f).set_index('date')['close']
    p.index = pd.to_datetime(p.index).normalize()
    return p


def compute_forward_returns(price_series: pd.Series, entry_date: pd.Timestamp,
                             horizons: list) -> dict:
    """
    Compute close-to-close returns for each horizon from entry_date.

    If the price is missing on entry_date (e.g. holiday), roll forward up to 3
    business days to find a fill price.  The exit is computed from the ACTUAL
    fill date (not the original requested date) so that a 20d return is always
    20 trading days — never 17-19d due to a rolled entry.
    """
    result = {f'fwd_{h}d': np.nan for h in horizons}
    if price_series.empty:
        return result

    # Determine actual fill date
    actual_entry = entry_date
    entry_price  = price_series.get(entry_date, np.nan)
    if np.isnan(entry_price) or entry_price == 0:
        for offset in [1, 2, 3]:
            candidate   = entry_date + pd.offsets.BDay(offset)
            ep          = price_series.get(candidate, np.nan)
            if not np.isnan(ep) and ep != 0:
                entry_price  = ep
                actual_entry = candidate   # exit is measured from actual fill, not original date
                break

    if np.isnan(entry_price) or entry_price == 0:
        return result

    for h in horizons:
        exit_date  = actual_entry + pd.offsets.BDay(h)   # always h days from actual entry
        exit_price = price_series.get(exit_date, np.nan)
        if not np.isnan(exit_price) and exit_price != 0:
            result[f'fwd_{h}d'] = exit_price / entry_price - 1
    return result


# Compute returns per unique (ticker, entry_date) pair then merge back
pairs = df_total[['BESTTICKER','entry_date']].drop_duplicates().copy()
print(f'Computing forward returns for {len(pairs):,} unique (ticker, date) pairs...')

return_records = []
price_cache    = {}

for _, row in pairs.iterrows():
    t = row['BESTTICKER']
    d = row['entry_date']
    if t not in price_cache:
        price_cache[t] = load_price_series(t)
        if len(price_cache) > 500:
            oldest = next(iter(price_cache))
            del price_cache[oldest]
    rets = compute_forward_returns(price_cache[t], d, HORIZONS)
    rets['BESTTICKER'] = t
    rets['entry_date'] = d
    return_records.append(rets)

returns_df = pd.DataFrame(return_records)
df_total   = df_total.merge(returns_df, on=['BESTTICKER','entry_date'], how='left')

fwd_cols = [f'fwd_{h}d' for h in HORIZONS]
coverage  = df_total[fwd_cols].notna().mean()
print('\nForward return coverage (% of events with valid price):')
print(coverage.round(3).to_string())


Computing forward returns for 376,351 unique (ticker, date) pairs...



Forward return coverage (% of events with valid price):
fwd_1d     0.356
fwd_3d     0.359
fwd_5d     0.361
fwd_10d    0.356
fwd_20d    0.350


## 2.8 Save the Augmented Signal File

Write the Total slice enriched with `entry_date`, universe flags, and forward returns to Parquet.  
Downstream notebooks load this file — it is the single source of truth for the backtest.

In [11]:
df_total.to_parquet(AUGMENTED_PQ, index=False, engine='pyarrow')
print(f'Saved augmented signal file → {AUGMENTED_PQ}')
print(f'Shape: {df_total.shape}')

# Quick summary of what we have
print('\n=== Ready for backtesting ===')
for univ in ['in_sp500','in_sp1500','in_ru3k']:
    sub = df_total[df_total[univ]]
    cov = sub[fwd_cols].notna().mean()
    print(f"\n{univ.upper()}  ({len(sub):,} events, {sub['BESTTICKER'].nunique():,} tickers)")
    print(cov.round(3).to_string())

Saved augmented signal file → /Users/chaithanyapakala/Downloads/NLP_FinalProject/data/signals_with_returns.parquet
Shape: (376790, 609)

=== Ready for backtesting ===

IN_SP500  (32,203 events, 783 tickers)
fwd_1d     0.839
fwd_3d     0.849
fwd_5d     0.854
fwd_10d    0.848
fwd_20d    0.826



IN_SP1500  (79,906 events, 1,465 tickers)
fwd_1d     0.954
fwd_3d     0.964
fwd_5d     0.971
fwd_10d    0.963
fwd_20d    0.936



IN_RU3K  (214,124 events, 7,372 tickers)
fwd_1d     0.617
fwd_3d     0.622
fwd_5d     0.625
fwd_10d    0.616
fwd_20d    0.605
